# Section 2: Planner, tools, and routing

Owner: Guna

This is my dev notebook for section 2 of the final notebook. It shows three things from the rubric:

1. **The agent plans its research steps.** For a stock symbol, the planner decides which sources to check and in what order.
2. **The agent uses tools dynamically.** Each planned step calls a real tool (yfinance for earnings and prices, NewsAPI or yfinance for news).
3. **Routing.** Each step is sent to the right specialist gatherer, and only the steps in the plan are run.

All the logic lives in `src/investment_committee/`. This notebook only calls it and shows the results.

## Setup

Run `pip install -e .` from the repo root first (or `make setup`).

The notebook defaults to `LIVE = False`, which uses a few small sample records and runs without internet. Set it to `True` when you want to call the real tools. The OpenAI planner only runs if `OPENAI_API_KEY` is set in `.env`. Without a key it falls back to the rule-based planner, and the notebook shows that too.

In [ ]:
from contextlib import closing
from pathlib import Path

import pandas as pd

import investment_committee
from investment_committee.agents.planner import ResearchPlan, ResearchTask, plan_research
from investment_committee.models import Evidence, MemoryNote, Run
from investment_committee.storage import load_evidence, load_model_calls, open_db
from investment_committee.workflows.chaining import RawArticle
from investment_committee.workflows.routing import default_gatherers, route_plan
from investment_committee.workflows.runner import run_research

LIVE = False
SYMBOL = "AAPL"

# A separate database file for this notebook, so it does not mix with the app's runs.
PROJECT_ROOT = Path(investment_committee.__file__).resolve().parents[2]
DEMO_DB = PROJECT_ROOT / "data" / "notebook_demo.db"

pd.set_option("display.max_colwidth", 120)

In [ ]:
def plan_table(plan):
    """Show a research plan as a small table."""
    rows = []
    for number, task in enumerate(plan.tasks, start=1):
        rows.append({"Step": number, "Source": task.source, "Reason": task.reason})
    return pd.DataFrame(rows)


def results_table(items):
    """Show gathered Evidence or news articles as a table."""
    rows = []
    for item in items:
        if isinstance(item, Evidence):
            rows.append({"Title": item.title, "Content": item.content, "As of": item.as_of})
        elif isinstance(item, RawArticle):
            rows.append({"Title": item.title, "Source": item.source, "Published": item.published})
    return pd.DataFrame(rows)


def sample_gatherer(source):
    """Offline stand-in for a real tool: returns one fixed record."""

    def gather(run):
        return [
            Evidence(
                run_id=run.run_id,
                symbol=run.symbol,
                source=source,
                title=f"Sample {source} record",
                content="Offline sample data, used when LIVE is False.",
                as_of="2026-10-01",
                reference="notebook sample",
            )
        ]

    return gather


if LIVE:
    gatherers = default_gatherers()
else:
    gatherers = {source: sample_gatherer(source) for source in ("earnings", "news", "market")}

## 2.1 The planner makes a plan

First the rule-based planner. It always gives the same plan, in a fixed order: earnings, news, then market. Each step has a short reason.

In [ ]:
run = Run(symbol=SYMBOL)
rules = plan_research(SYMBOL, run.run_id, provider="rules")
plan_table(rules.plan)

The plan is not fixed forever, though. If the last run for this stock left a note saying something was missing, the planner moves that source to the front. Here we pretend the previous run had no news evidence.

(Section 6 shows this with two real runs and a note saved in SQLite. Here it is just a quick look.)

In [ ]:
old_note = MemoryNote(
    run_id="earlier-run",
    symbol=SYMBOL,
    confidence=0.4,
    open_gaps=["no news evidence in the pool"],
    lesson="Check the news first next time.",
)
with_memory = plan_research(SYMBOL, run.run_id, provider="rules", previous_note=old_note)
print("Lesson from the last run:", with_memory.plan.memory_lesson)
plan_table(with_memory.plan)

News is now step 1, and its reason says it is revisiting the gap from the last run.

## 2.2 The OpenAI planner

The default planner asks an OpenAI model to choose one to three sources and explain why. The answer has to match a fixed format (a list of tasks with a source and a reason), so a bad answer is rejected instead of breaking the run.

If there is no API key, or the call fails, the planner falls back to the rule-based plan and records why. Either way we still get a plan.

In [ ]:
model_plan = plan_research(SYMBOL, run.run_id, provider="openai")

print("Planner asked for:", model_plan.provider_requested)
print("Planner used:     ", model_plan.provider_used)
if model_plan.fallback_reason:
    print("Fell back because:", model_plan.fallback_reason)
if model_plan.model_call:
    call = model_plan.model_call
    print("Model:            ", call.model)
    print("Tokens in / out:  ", call.prompt_tokens, "/", call.completion_tokens)
    print("Time (ms):        ", call.latency_ms)

plan_table(model_plan.plan)

The model can choose fewer sources than the rule-based plan, or a different order. That is the point of letting it plan: the next step (routing) only runs what the plan asks for.

## 2.3 Routing: each step goes to its own gatherer

The router keeps a simple map from source to tool:

In [ ]:
for source, gatherer in default_gatherers().items():
    print(f"{source:9} -> {gatherer.__module__}.{gatherer.__name__}")

To show that routing really follows the plan, here is a plan with only a market step. We wrap each gatherer so it records when it is called. Only the market tool should run.

In [ ]:
called = []


def record_calls(source, gatherer):
    """Wrap a gatherer so we can see which ones the router called."""

    def gather(run):
        called.append(source)
        return gatherer(run)

    return gather


market_only = ResearchPlan(
    symbol=SYMBOL,
    tasks=[ResearchTask(source="market", reason="Only check the price this time.")],
)
watched = {source: record_calls(source, gatherer) for source, gatherer in gatherers.items()}
routed = route_plan(Run(symbol=SYMBOL), market_only, gatherers=watched)

print("Gatherers called:", called)
print("Results returned for:", list(routed))

## 2.4 The tools return real data

Now we route the full rule-based plan, so all three tools run. Earnings and market come back as `Evidence` records that later stages can cite. News comes back as raw articles, because Mina's news chain still has to clean and classify them before they become evidence.

In [ ]:
full_run = Run(symbol=SYMBOL)
full_plan = plan_research(SYMBOL, full_run.run_id, provider="rules").plan
gathered = route_plan(full_run, full_plan, gatherers=gatherers)

for source, items in gathered.items():
    print(f"{source}: {len(items)} records")

In [ ]:
results_table(gathered["earnings"])

In [ ]:
results_table(gathered["market"])

In [ ]:
results_table(gathered["news"]).head(5)

A note on news: in early October 2026 Yahoo's ticker news feed started returning nothing. The news tool now tries three sources in order and uses the first one that returns articles:

1. yfinance ticker news
2. NewsAPI (only if `NEWSAPI_KEY` is set)
3. yfinance search, which only has headlines

This is a good example of why the agent should not depend on one free data source.

## 2.5 The whole thing in one call

`run_research` does all of the above in one step: read the last memory note, plan, route, gather, and save the run to SQLite. If one source has no data (for example an ETF has no earnings), the run still finishes and says which source was missing.

In [ ]:
snapshot = run_research(
    SYMBOL,
    gatherers=gatherers,
    database_path=DEMO_DB,
    planner_provider="openai",
)

print("Run id:          ", snapshot.run.run_id)
print("Planner used:    ", snapshot.planner_provider_used, snapshot.planner_model or "")
print("Plan:            ", [task.source for task in snapshot.plan.tasks])
print("Saved evidence:  ", snapshot.saved_evidence_count)
for source, message in snapshot.gather_errors.items():
    print(f"Warning ({source}):", message)
if snapshot.memory_error:
    print("Memory warning:", snapshot.memory_error)

Finally, we read the run back from SQLite. This is how the later stages (Bull, Bear, and the evaluator) get the evidence: by run id, from the database, not from this notebook's variables.

In [ ]:
with closing(open_db(DEMO_DB)) as conn:
    saved = load_evidence(conn, snapshot.run.run_id)
    calls = load_model_calls(conn, snapshot.run.run_id)

print(f"{len(saved)} evidence records and {len(calls)} model call(s) saved for this run.")
results_table(saved)

## What this section shows

- **Planning:** the planner picks the sources and their order, and changes the order when the last run reported a gap.
- **Tool use:** every planned step calls a real data tool, and a tool with no data does not stop the run.
- **Routing:** each step goes to its own gatherer, and steps that are not in the plan are skipped.
- **Fallbacks:** no OpenAI key means the rule-based plan; no Yahoo news means NewsAPI or search headlines.